# RXNMapper + RDChiral 教学示例：从原始反应到模板回推

这份 notebook 只保留最值得先掌握的一条主线：

1. 输入一个 **没有 atom mapping 的 reaction SMILES**
2. 用 **RXNMapper** 自动补齐原子映射
3. 用 **RDChiral** 从带映射反应中抽取 reaction template
4. 把模板应用到目标产物上，观察回推出的候选前体

目录已经配好一键环境脚本：

- `bash teaching_demos/reaction_template_tutorial/setup_env.sh`
- `bash teaching_demos/reaction_template_tutorial/launch_notebook.sh`

环境会统一创建到 `envs/reaction_template_tutorial_envs`。


## 这份 notebook 建议这样理解

- 先把这份 notebook 当作一条主干流程，抓住每一步的输入和输出
- 再区分两个工具的职责：`RXNMapper` 负责 **atom mapping**，`RDChiral` 负责 **模板抽取与模板应用**
- 最后用相似产物做泛化测试，观察模板为什么能迁移，以及它的边界在哪里

本示例选择的是一个 **Grignard 加成** 教学案例，因为模板结构清楚、回推结果也直观。


In [1]:
from pathlib import Path
import sys
import warnings

import pandas as pd
from IPython.display import HTML, display


def locate_tutorial_dir(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    if here.is_file():
        here = here.parent

    for candidate in [here, *here.parents]:
        direct = candidate / "teaching_demos" / "1.reaction_template_tutorial"
        if direct.exists():
            return direct
        if (candidate / "RXNMapper_RDChiral_教学示例.ipynb").exists():
            return candidate
    raise FileNotFoundError("无法定位 teaching_demos/reaction_template_tutorial 目录。")


warnings.filterwarnings("ignore", message="pkg_resources is deprecated.*")

TUTORIAL_DIR = locate_tutorial_dir()
if str(TUTORIAL_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_DIR))

from tutorial_utils import (
    draw_molecule_grid_svg,
    draw_reaction_svg,
    ensure_source_repos_on_path,
    environment_report,
    evaluate_template_targets,
    find_project_root,
    format_project_path,
    route_gallery_html,
    run_template_workflow,
    split_smiles_side,
)

PROJECT_ROOT = find_project_root(TUTORIAL_DIR)
REPO_PATHS = ensure_source_repos_on_path(PROJECT_ROOT)

print(f"PROJECT_ROOT = {format_project_path(PROJECT_ROOT, PROJECT_ROOT)}")
print(f"TUTORIAL_DIR = {format_project_path(TUTORIAL_DIR, PROJECT_ROOT)}")


PROJECT_ROOT = project/
TUTORIAL_DIR = project/teaching_demos/1.reaction_template_tutorial


## 0. 环境自检

开始前先确认三件事：

- Notebook 确实运行在我们准备好的环境里
- `source_repos/rxnmapper` 与 `source_repos/rdchiral` 都能被找到
- `rdkit / torch / transformers` 这些核心依赖已经可用

如果这里有缺项，后面出错时就很难分辨到底是“化学逻辑问题”还是“环境问题”。


In [2]:
pd.set_option("display.max_colwidth", 120)
display(environment_report(PROJECT_ROOT))


,组件,版本,位置
0,python,3.12.13,project/envs/reaction_template_tutorial_envs/bin/python3.12
1,project_root,-,project/
2,tutorial_dir,-,project/teaching_demos/reaction_template_tutorial
3,rdchiral_repo,-,project/source_repos/rdchiral
4,rxnmapper_repo,-,project/source_repos/rxnmapper
5,pandas,3.0.1,project/envs/reaction_template_tutorial_envs/lib/python3.12/site-packages/pandas/__init__.py
6,rdkit,2025.09.6,project/envs/reaction_template_tutorial_envs/lib/python3.12/site-packages/rdkit/__init__.py
7,torch,2.10.0+cpu,project/envs/reaction_template_tutorial_envs/lib/python3.12/site-packages/torch/__init__.py
8,transformers,4.57.6,project/envs/reaction_template_tutorial_envs/lib/python3.12/site-packages/transformers/__init__.py
9,rxnmapper,0.4.3,project/source_repos/rxnmapper/rxnmapper/__init__.py


## 1. 从一个未映射反应开始

我们先输入最原始的 reaction SMILES。此时它只表达“谁和谁反应生成了什么”，**没有 atom mapping**，所以模板抽取还无法直接开始。

本例对应的是一个芳香醛与甲基格氏试剂的加成反应。


In [3]:
reaction_smiles_unmapped = "CCOCC.C[Mg+].O=Cc1ccc(F)cc1Cl.[Br-]>>CC(O)c1ccc(F)cc1Cl"
target_product_smiles = "CC(O)c1ccc(F)cc1Cl"

print("未映射 reaction SMILES:")
print(reaction_smiles_unmapped)

reactant_smiles = split_smiles_side(reaction_smiles_unmapped.split(">>")[0])
product_smiles = split_smiles_side(reaction_smiles_unmapped.split(">>")[1])

display(HTML(draw_reaction_svg(reaction_smiles_unmapped, use_smiles=True)))
display(
    HTML(
        draw_molecule_grid_svg(
            reactant_smiles + product_smiles,
            legends=[f"Reactant {idx + 1}" for idx in range(len(reactant_smiles))]
            + [f"Product {idx + 1}" for idx in range(len(product_smiles))],
            mols_per_row=3,
        )
    )
)


未映射 reaction SMILES:
CCOCC.C[Mg+].O=Cc1ccc(F)cc1Cl.[Br-]>>CC(O)c1ccc(F)cc1Cl


## 2. 用 RXNMapper 自动补 atom mapping

这里最关键的理解是：

- 原始 reaction SMILES 只告诉你“分子前后变了”
- 模板算法还需要知道“**哪一个原子**从反应物侧对应到了产物侧”

`RXNMapper()` 第一次初始化时会加载 transformer 模型，所以第一次运行通常会比后续慢一些。


In [4]:
from rxnmapper import RXNMapper

rxn_mapper = RXNMapper()
mapping_result = rxn_mapper.get_attention_guided_atom_maps([reaction_smiles_unmapped])[0]

mapped_reaction_smiles = mapping_result["mapped_rxn"]
mapping_confidence = mapping_result["confidence"]
print("映射后的 reaction SMILES:", mapped_reaction_smiles)
print("映射置信度:", round(mapping_confidence, 4))
display(
    pd.DataFrame(
        [
            {"字段": "mapping_confidence", "值": round(mapping_confidence, 4)},
            {"字段": "mapped_rxn", "值": mapped_reaction_smiles},
        ]
    )
)


映射后的 reaction SMILES: CCOCC.[CH:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]([F:8])[cH:9][c:10]1[Cl:11].[Br-].[Mg+][CH3:1]>>[CH3:1][CH:2]([OH:3])[c:4]1[cH:5][cH:6][c:7]([F:8])[cH:9][c:10]1[Cl:11]
映射置信度: 0.8947


,字段,值
0,mapping_confidence,0.8947
1,mapped_rxn,CCOCC.[CH:2](=[O:3])[c:4]1[cH:5][cH:6][c:7]([F:8])[cH:9][c:10]1[Cl:11].[Br-].[Mg+][CH3:1]>>[CH3:1][CH:2]([OH:3])[c:4...


In [5]:
mapped_reactants, mapped_products = mapped_reaction_smiles.split(">>")

display(HTML(draw_reaction_svg(mapped_reaction_smiles, use_smiles=True)))
display(
    HTML(
        draw_molecule_grid_svg(
            split_smiles_side(mapped_reactants) + split_smiles_side(mapped_products),
            legends=[f"Mapped reactant {idx + 1}" for idx in range(len(split_smiles_side(mapped_reactants)))]
            + [f"Mapped product {idx + 1}" for idx in range(len(split_smiles_side(mapped_products)))],
            annotate_atom_maps=True,
            mols_per_row=3,
        )
    )
)


上面的第二张图可以重点观察两件事：

- 同一个 atom map 编号，表示反应前后是“同一个原子”
- 真正参与转化的，是那些邻接关系或官能团环境发生变化的编号

一旦看懂这个映射视角，就更容易理解为什么模板抽取可以成立。


## 3. 用 RDChiral 抽取 reaction template

`RDChiral` 需要的不是整条未处理反应，而是一个带映射的反应记录。最常用的字段只有三个：

- `_id`
- `reactants`
- `products`

抽取结果里最重要的字段是 `reaction_smarts`。它描述了“反应中心应该怎样变化”。


In [6]:
from rdchiral.template_extractor import extract_from_reaction

reaction_record = {
    "_id": "demo_grignard_001",
    "reactants": mapped_reactants,
    "products": mapped_products,
}

template = extract_from_reaction(reaction_record)
reaction_template_smarts = template["reaction_smarts"]

display(
    pd.DataFrame(
        [
            {"字段": "reaction_id", "值": template["reaction_id"]},
            {"字段": "reaction_smarts", "值": reaction_template_smarts},
            {"字段": "products 模板", "值": template["products"]},
            {"字段": "reactants 模板", "值": template["reactants"]},
            {"字段": "necessary_reagent", "值": template["necessary_reagent"] or "(空)"},
        ]
    )
)


,字段,值
0,reaction_id,demo_grignard_001
1,reaction_smarts,[CH3;D1;+0:1]-[CH;D3;+0:3](-[OH;D1;+0:2])-[c:4]>>[CH3;D1;+0:1]-[Mg+].[O;H0;D1;+0:2]=[CH;D2;+0:3]-[c:4]
2,products 模板,[CH3;D1;+0:1]-[CH;D3;+0:3](-[OH;D1;+0:2])-[c:4]
3,reactants 模板,[CH3;D1;+0:1]-[Mg+].[O;H0;D1;+0:2]=[CH;D2;+0:3]-[c:4]
4,necessary_reagent,(空)


In [7]:
print("提取出的 retrosynthesis 模板:")
print(reaction_template_smarts)

display(HTML(draw_reaction_svg(reaction_template_smarts, use_smiles=False)))


提取出的 retrosynthesis 模板:
[CH3;D1;+0:1]-[CH;D3;+0:3](-[OH;D1;+0:2])-[c:4]>>[CH3;D1;+0:1]-[Mg+].[O;H0;D1;+0:2]=[CH;D2;+0:3]-[c:4]


这里要特别注意：`reaction_smarts` 在当前工作流里默认是 **retrosynthesis 方向**。

也就是说，它更适合回答的是：

- “如果我已经有目标产物，这个模板会把它回推成什么前体？”

而不是直接按正向反应去做产物预测。


## 4. 把模板应用到目标产物上

现在把模板当作一个“逆合成规则”，输入目标产物，看看 `RDChiral` 会回推出哪些候选前体。


In [8]:
from rdchiral.main import rdchiralReaction, rdchiralReactants, rdchiralRun

rxn = rdchiralReaction(reaction_template_smarts)
outcomes = sorted(rdchiralRun(rxn, rdchiralReactants(target_product_smiles)))

print("目标产物:")
print(target_product_smiles)
print()
print("候选前体:")
for idx, outcome in enumerate(outcomes, start=1):
    print(f"{idx}. {outcome}")

display(
    route_gallery_html(
        [
            {
                "label": "原始目标分子",
                "target_product_smiles": target_product_smiles,
                "first_outcome": outcomes[0] if outcomes else "",
                "match_count": len(outcomes),
            }
        ]
    )
)


目标产物:
CC(O)c1ccc(F)cc1Cl

候选前体:
1. O=Cc1ccc(F)cc1Cl.[CH3][Mg+]


从这个结果可以进一步理解模板的化学含义：

- 产物中的仲醇中心，被模板回推成了对应的芳香醛
- 同时保留了甲基格氏试剂这一类典型亲核加成前体

也就是说，模板并不是“背答案”，而是在重建反应中心的局部变换规则。


## 5. 看模板对相似目标分子的泛化边界

模板抽取之后，不只是能回推原始案例，还可以尝试回推一组相似分子。

这一步可以帮助你理解两个问题：

- 模板为什么有泛化能力
- 模板为什么又不是无限泛化


In [9]:
generalization_targets = [
    ("原始案例", "CC(O)c1ccc(F)cc1Cl"),
    ("对位 Br 类似物", "CC(O)c1ccc(F)cc1Br"),
    ("二氯类似物", "CC(O)c1ccc(Cl)cc1Cl"),
    ("芳环位点重排", "CC(O)c1cccc(F)c1Cl"),
    ("带甲基取代", "CC(O)c1ccc(F)c(C)c1Cl"),
    ("带甲氧基取代", "CC(O)c1ccc(F)c(OC)c1Cl"),
]

generalization_df = evaluate_template_targets(reaction_template_smarts, generalization_targets)
display(generalization_df[["label", "target_product_smiles", "match_count", "first_outcome"]])


,label,target_product_smiles,match_count,first_outcome
0,原始案例,CC(O)c1ccc(F)cc1Cl,1,O=Cc1ccc(F)cc1Cl.[CH3][Mg+]
1,对位 Br 类似物,CC(O)c1ccc(F)cc1Br,1,O=Cc1ccc(F)cc1Br.[CH3][Mg+]
2,二氯类似物,CC(O)c1ccc(Cl)cc1Cl,1,O=Cc1ccc(Cl)cc1Cl.[CH3][Mg+]
3,芳环位点重排,CC(O)c1cccc(F)c1Cl,1,O=Cc1cccc(F)c1Cl.[CH3][Mg+]
4,带甲基取代,CC(O)c1ccc(F)c(C)c1Cl,1,Cc1c(F)ccc(C=O)c1Cl.[CH3][Mg+]
5,带甲氧基取代,CC(O)c1ccc(F)c(OC)c1Cl,1,COc1c(F)ccc(C=O)c1Cl.[CH3][Mg+]


In [10]:
display(route_gallery_html(generalization_df.to_dict("records")))


## 6. 把主线封装成一个可复用接口

为了后续自己复用，目录里已经提供了 `run_template_workflow(...)`。它把下面四步串了起来：

1. unmapped reaction -> mapped reaction
2. mapped reaction -> template
3. template -> retrosynthetic outcomes
4. 返回一个结构化字典，方便后续展示或批处理


In [11]:
workflow_summary = run_template_workflow(
    unmapped_rxn_smiles=reaction_smiles_unmapped,
    target_product_smiles="CC(O)c1ccc(F)c(OC)c1Cl",
    reaction_id="demo_grignard_variant",
    mapper=rxn_mapper,
)

display(
    pd.DataFrame(
        [
            {"字段": "mapping_confidence", "值": round(workflow_summary["mapping_confidence"], 4)},
            {"字段": "reaction_template_smarts", "值": workflow_summary["reaction_template_smarts"]},
            {"字段": "target_product_smiles", "值": workflow_summary["target_product_smiles"]},
            {
                "字段": "first_outcome",
                "值": workflow_summary["outcomes"][0] if workflow_summary["outcomes"] else "(无匹配)",
            },
        ]
    )
)

display(
    route_gallery_html(
        [
            {
                "label": "函数式调用示例",
                "target_product_smiles": workflow_summary["target_product_smiles"],
                "first_outcome": workflow_summary["outcomes"][0] if workflow_summary["outcomes"] else "",
                "match_count": len(workflow_summary["outcomes"]),
            }
        ]
    )
)


,字段,值
0,mapping_confidence,0.8947
1,reaction_template_smarts,[CH3;D1;+0:1]-[CH;D3;+0:3](-[OH;D1;+0:2])-[c:4]>>[CH3;D1;+0:1]-[Mg+].[O;H0;D1;+0:2]=[CH;D2;+0:3]-[c:4]
2,target_product_smiles,CC(O)c1ccc(F)c(OC)c1Cl
3,first_outcome,COc1c(F)ccc(C=O)c1Cl.[CH3][Mg+]


## 7. 小结与思考题

到这里，你应该能完整说出这条主线：

1. `RXNMapper` 负责把反应写成“原子可追踪”的版本
2. `RDChiral` 负责从带映射反应里抽出模板
3. 这个模板默认用于 retrosynthesis，应当输入目标产物
4. 模板在相似骨架上通常可迁移，但迁移能力受上下文保留程度限制

可以继续思考下面几个问题：

- 如果 atom mapping 错了，模板会怎样受影响？
- 为什么模板能回推 `Br` 或 `OMe` 类似物？
- 哪些结构变化会让这个模板失效？
